# 2. Amazon Bedrock Knowledge Bases - Creating GraphRAG
- Tested in Amazon SageMaker AI - Notebook - JupyterLab environment.
- Kernel: conda_python3

![graphrag00](../img/graphrag-00-en.png)
The integration of Amazon Bedrock's Knowledge Bases with Amazon Neptune Analytics makes it easy to implement GraphRAG. GraphRAG extends traditional RAG technology by organizing relationships between documents as a graph, enabling more accurate and contextually relevant information extraction from complex documents. This technology automatically creates graphs that connect document chunks with entities and relationships found in unstructured data, providing more comprehensive and accurate responses without requiring graph expertise.

> For more details, refer to the AWS technical blog [Unstructured Data! Properly Utilizing with Amazon Bedrock](https://aws.amazon.com/ko/blogs/tech/bedrock-data-automation-graphrag/).

## 1. Setup

### Installing Required Libraries

In [ ]:
# Install required libraries (kernel restart required after installation)
!pip install -q boto3 --upgrade
!pip install -q awscli --upgrade

### Importing Libraries and Setting Up Session

In [23]:
# Import required libraries
import boto3
import json
import time
import os
import uuid
from datetime import datetime
import pandas as pd
from tqdm.notebook import tqdm

# Set AWS region
region = boto3.session.Session().region_name
print(f"Current AWS Region: {region}")

# Set up session and clients
session = boto3.session.Session(region_name=region)
bedrock = session.client('bedrock')
bedrock_runtime = session.client('bedrock-runtime')
bedrock_agent = session.client('bedrock-agent')
bedrock_agent_runtime = boto3.client('bedrock-agent-runtime')
s3 = session.client('s3')


Current AWS Region: us-west-2


In [4]:
print(boto3.__version__)

1.37.3


## 2. Creating KB - GraphRAG 
- [Note] KB - GraphRAG creation through boto3 and awscli is expected to be supported in Q1 2025.

<b> Select Knowledge Bases from the Amazon Bedrock console screen. </b>

![graphrag01](../img/graphrag-01.png)

<b> Click the Create button and select Knowledge Base with vector store. </b>

![graphrag03](../img/graphrag-02.png)

<b> Enter and select the following information: 
- Knowledge Base name: graphrag-workshop
- IAM permission: Select Create and use a new service role
- Choose data source: Select Amazon S3
Then click the Next button.
</b>

![graphrag03](../img/graphrag-03.png)

<b> Enter and select the following information: 
- Data source name: 10-q
- S3 URI: Enter or select the previously created S3 path s3://~~~/data/
Then click the Next button.
</b>

![graphrag04](../img/graphrag-04.png)

<b> Enter and select the following information: 
- Embeddings model: Titan Text Embeddings V2
- Vector store: Amazon Neptune Analytics (GraphRAG) - Preview
Then click the Next button.
</b>

![graphrag05](../img/graphrag-05.png)

<b> After reviewing the KB creation information, click the Create Knowledge Base button.
</b>

![graphrag06](../img/graphrag-06.png)

<b> After a few minutes, the KB - GraphRAG will be created, and you can check the following information:
- Knowledge Base ID

Save this information in a variable for use in the next steps.
</b>

![graphrag07](../img/graphrag-07.png)

In [17]:
# Enter the kb_id value for your created KB - GraphRAG
# Example: kb_id = "WSPOHGBATW"


kb_id = "your environment's kb_id"


### Waiting for KB Creation to Complete

<b> Perform Data Sync in the Data source:
- Select the Data source (10-q).
- Click the Sync button.
Wait a few minutes for the Data Sync to complete. When completed, the Status will show as Available.
</b>

![graphrag09](../img/graphrag-09.png)

---

## 3. Querying KB - GraphRAG
- Using Claude 3.5 Sonnet

In [34]:
# Verify kb_id value

print(kb_id)

WSPOHGBATW


In [37]:
def query_knowledge_base(query_text, model_id, max_tokens=1000):
    try:
        # Configure search request
        retrieve_response = bedrock_agent_runtime.retrieve(
            knowledgeBaseId=kb_id,
            retrievalQuery={
                'text': query_text
            },
            retrievalConfiguration={
                'vectorSearchConfiguration': {
                    'numberOfResults': 3,
                    'overrideSearchType': 'SEMANTIC'                    
                }
            }
        )
        
        # Check search results
        retrieved_results = retrieve_response.get('retrievalResults', [])
        
        if not retrieved_results:
            print("No search results found.")
            return None
        
        # Use search results as context
        context = ""
        for i, result in enumerate(retrieved_results):
            content = result['content']['text']
            source = result.get('location', {}).get('s3Location', {}).get('uri', 'Unknown source')
            score = result.get('score', 0)
            
            context += f"\n\nReference Document {i+1} (Relevance score: {score}):\n{content}\n"
            print(f"Search result {i+1}: Relevance score {score}")
        
        # Configure query for Claude
        prompt = f"""
Please answer the user's question. Use the following reference documents:

{context}

User question: {query_text}

Answer:
"""
        
        # Send query request to Claude
        response = bedrock_runtime.converse(
            modelId=modelId,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "text": prompt
                        }
                    ]
                }
            ],
            inferenceConfig={
                "maxTokens": max_tokens,
                "temperature": 0.7,
                "topP": 0.9
            }
        )
        
        # Extract and return response
        answer = response['output']['message']['content'][0]['text']
        return answer
        
    except Exception as e:
        print(f"Error occurred during query execution: {e}")
        return None


### Testing User Queries

In [40]:
# Run query examples
test_queries = [
    "How has Amazon's total net sales changed over time?",
    "How has the increase in Amazon's operating expenses affected the profitability of each business segment and overall financial performance, and what correlations exist with other financial metrics?"
]

modelId="us.anthropic.claude-3-5-sonnet-20241022-v2:0"    # Claude 3.5 Sonnet v2 (cross-inference)
#modelId="us.anthropic.claude-3-7-sonnet-20250219-v1:0"    # Claude 3.7 Sonnet (cross-inference)

for query in test_queries:
    print(f"\nQuestion: {query}")
    answer = query_knowledge_base(query, modelId)
    if answer:
        print(f"\nAnswer:\n{answer}")
    print("-" * 80)



Question: How has Amazon's total net sales changed over time?
Search result 1: Relevance score 1.045573
Search result 2: Relevance score 1.035476
Search result 3: Relevance score 1.02006

Answer:
Based on the provided data, Amazon's total net sales (Total net sales) over time can be summarized chronologically as follows:

Q3 2021: $110.8 billion
Q3 2022: $127.1 billion
Q3 2023: $143.1 billion

Analyzing this data:
- Between Q3 2021 and Q3 2022: approximately 14.7% increase
- Between Q3 2022 and Q3 2023: approximately 12.6% increase

Overall, Amazon's total net sales show a steady growth trend. This growth is attributed to increases in both Net product sales and Net service sales, with the service sector showing particularly notable growth.

For Q4 2022, Amazon provided guidance expecting sales between $140.0 billion and $148.0 billion.
--------------------------------------------------------------------------------

Question: How has the increase in Amazon's operating expenses affecte

---

## 5. Clean-up

In [ ]:
# Set resource IDs/names created in the workshop

kb_id = "Enter your Knowledge Base ID here"
bucket_name = "Enter your S3 bucket name here"


# Initialize AWS clients
import boto3
region = boto3.session.Session().region_name
bedrock_agent = boto3.client('bedrock-agent', region_name=region)
s3 = boto3.client('s3', region_name=region)
iam = boto3.client('iam', region_name=region)

print(f"Variables set for resource cleanup: region: {region}")

In [ ]:
# Delete Knowledge Base
try:
    print(f"Deleting Knowledge Base: {kb_id}")
    response = bedrock_agent.delete_knowledge_base(knowledgeBaseId=kb_id)
    print(f"Knowledge Base deletion request successfully submitted.")
    print("Status: Deletion in progress in the background...")
except Exception as e:
    print(f"Error occurred while deleting Knowledge Base: {e}")

In [ ]:
# Delete all objects in S3 bucket and then delete the bucket
import time

try:
    # Delete all objects
    print(f"Deleting all objects in S3 bucket: {bucket_name}")

    # List and delete all objects
    paginator = s3.get_paginator('list_objects_v2')
    object_count = 0

    for page in paginator.paginate(Bucket=bucket_name):
        if 'Contents' in page:
            objects = [{'Key': obj['Key']} for obj in page['Contents']]
            s3.delete_objects(Bucket=bucket_name, Delete={'Objects': objects})
            object_count += len(objects)

    print(f"{object_count} objects deleted")

    # Delete versioned objects if needed
    try:
        paginator = s3.get_paginator('list_object_versions')
        version_count = 0

        for page in paginator.paginate(Bucket=bucket_name):
            delete_list = []

            # Delete versions
            if 'Versions' in page:
                delete_list.extend([{'Key': obj['Key'], 'VersionId': obj['VersionId']} for obj in page['Versions']])

            # Remove delete markers
            if 'DeleteMarkers' in page:
                delete_list.extend([{'Key': obj['Key'], 'VersionId': obj['VersionId']} for obj in page['DeleteMarkers']])

            if delete_list:
                s3.delete_objects(Bucket=bucket_name, Delete={'Objects': delete_list})
                version_count += len(delete_list)

        if version_count > 0:
            print(f"{version_count} object versions/delete markers removed")
    except Exception as e:
        print(f"Error during version removal (can be ignored): {e}")

    # Wait briefly (for all objects to be deleted)
    time.sleep(3)

    # Delete bucket
    print(f"Deleting S3 bucket: {bucket_name}")
    s3.delete_bucket(Bucket=bucket_name)
    print(f"S3 bucket deleted: {bucket_name}")

except Exception as e:
    print(f"Error occurred while deleting S3 bucket: {e}")